# DAAD Degree Scraper

A webscraper to scrape the degree programmes on the DAAD website and then save them in a dataset folder organised by the locations.

## 1.Setup

### 1.1 Installing Dependencies

In [ ]:
pip install pandas
pip install requests 
pip install beautifulsoup4
pip install openpyxl

SyntaxError: invalid syntax (2665302486.py, line 1)

In [2]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
import math

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/114.0 Safari/537.36",
}

output_dir = "datasets"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/bachelor", exist_ok=True)
os.makedirs(f"{output_dir}/master", exist_ok=True)

print("Setup complete.")

Setup complete.


In [3]:
base_url = "https://www.daad.de/en/studying-in-germany/universities/all-degree-programmes/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/114.0 Safari/537.36",
}

### 1.3 Scraper Function

In [4]:
def scrape_degree_programmes(base_url, headers, lastpage=None):
    all_programmes = []

    # using beautifulsoup to parse the first page and get total count of degree programmes
    resp = requests.get(f"{base_url}{'&' if '?' in base_url else '?'}hec-p=1&hec-limit=100", headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")

    count_h2 = soup.find("h2", class_="qa-db-result-count u-size-20 u-font-semibold mb-12 mr-16")
    count_text = count_h2.get_text(strip=True).split()[0].replace(",", "")
    if not count_text.isdigit():
        print("No programmes found.")
        return pd.DataFrame()
    total = int(count_text)
    total_pages = math.ceil(total / 100)
    print(f"Found {total} programmes → {total_pages} pages to scrape.")

    for page in range(1, lastpage+1 if lastpage else total_pages + 1):
        # Reuse the already-parsed soup for page 1, fetch fresh for the rest
        if page > 1:
            resp = requests.get(f"{base_url}{'&' if '?' in base_url else '?'}hec-p={page}&hec-limit=100", headers=headers)
            soup = BeautifulSoup(resp.text, "html.parser")

        print(f"Page {page}/{lastpage if lastpage else total_pages}...")
        items = [li for li in soup.find_all("li") if "qa-db-result-item" in " ".join(li.get("class", []))]
        if not items:
            break

        for li in items:
            h3 = li.find("h3")
            spans = h3.find_all("span") if h3 else []
            university = spans[0].get_text(strip=True) if len(spans) > 0 else None
            programme = spans[1].get_text(strip=True) if len(spans) > 1 else None

            dl = li.find("dl", class_="my-0 u-size-18 row items-grid")
            degree_type, location = None, None
            if dl:
                children = dl.find_all(recursive=False)
                if len(children) >= 3:
                    degree_type = children[0].get_text(strip=True)
                    if degree_type and degree_type.startswith("Degree:"):
                        degree_type = degree_type.replace("Degree:", "").strip()

                    loc_parts = children[2].find_all(recursive=False)
                    if len(loc_parts) >= 2:
                        location = loc_parts[1].get_text(strip=True)

            a = li.find("a", href=True)
            detail_url = "https://www.daad.de" + a["href"] if a else None

            all_programmes.append({
                "University": university,
                "Programme": programme,
                "Degree Type": degree_type,
                "Location": location,
                "Detail URL": detail_url
            })

        time.sleep(1)


    df = pd.DataFrame(all_programmes)
    df = df.drop_duplicates(subset="Detail URL").reset_index(drop=True)
    print(f"Done. {len(df)} unique programmes collected.")
    return df


In [5]:
#test the function
test = scrape_degree_programmes(base_url, headers, lastpage=3)

Found 22711 programmes → 228 pages to scrape.
Page 1/3...
Page 2/3...
Page 3/3...
Done. 300 unique programmes collected.


In [6]:
test, test.describe(), test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   University   300 non-null    object
 1   Programme    300 non-null    object
 2   Degree Type  300 non-null    object
 3   Location     300 non-null    object
 4   Detail URL   300 non-null    object
dtypes: object(5)
memory usage: 11.8+ KB


(                                   University  \
 0                  Technische Hochschule Köln   
 1    Mittweida University of Applied Sciences   
 2      Leipzig University of Applied Sciences   
 3    University of Applied Sciences Fresenius   
 4                      University of Würzburg   
 ..                                        ...   
 295                      Frankfurt University   
 296                     Heidelberg University   
 297                Humboldt University Berlin   
 298                     University of Leipzig   
 299                          Mainz University   
 
                                Programme Degree Type  \
 0        3D Animation for Film and Games      Master   
 1                            3D Printing    Bachelor   
 2    3D Printing and Functional Surfaces    Bachelor   
 3                Academic Speech Therapy    Bachelor   
 4                Academic Speech Therapy    Bachelor   
 ..                                   ...         ...   

## 2. Scraping the Degrees

In [7]:
#test the function
programmes = scrape_degree_programmes(base_url, headers, lastpage=None)
programmes.head()

Found 22711 programmes → 228 pages to scrape.
Page 1/228...
Page 2/228...
Page 3/228...
Page 4/228...
Page 5/228...
Page 6/228...
Page 7/228...
Page 8/228...
Page 9/228...
Page 10/228...
Page 11/228...
Page 12/228...
Page 13/228...
Page 14/228...
Page 15/228...
Page 16/228...
Page 17/228...
Page 18/228...
Page 19/228...
Page 20/228...
Page 21/228...
Page 22/228...
Page 23/228...
Page 24/228...
Page 25/228...
Page 26/228...
Page 27/228...
Page 28/228...
Page 29/228...
Page 30/228...
Page 31/228...
Page 32/228...
Page 33/228...
Page 34/228...
Page 35/228...
Page 36/228...
Page 37/228...
Page 38/228...
Page 39/228...
Page 40/228...
Page 41/228...
Page 42/228...
Page 43/228...
Page 44/228...
Page 45/228...
Page 46/228...
Page 47/228...
Page 48/228...
Page 49/228...
Page 50/228...
Page 51/228...
Page 52/228...
Page 53/228...
Page 54/228...
Page 55/228...
Page 56/228...
Page 57/228...
Page 58/228...
Page 59/228...
Page 60/228...
Page 61/228...
Page 62/228...
Page 63/228...
Page 64/228...
Pag

,University,Programme,Degree Type,Location,Detail URL
0,Technische Hochschule Köln,3D Animation for Film and Games,Master,Köln,https://www.daad.de/en/studying-in-germany/uni...
1,Mittweida University of Applied Sciences,3D Printing,Bachelor,Mittweida,https://www.daad.de/en/studying-in-germany/uni...
2,Leipzig University of Applied Sciences,3D Printing and Functional Surfaces,Bachelor,Leipzig,https://www.daad.de/en/studying-in-germany/uni...
3,University of Applied Sciences Fresenius,Academic Speech Therapy,Bachelor,"Idstein, Frankfurt am Main",https://www.daad.de/en/studying-in-germany/uni...
4,University of Würzburg,Academic Speech Therapy,Bachelor,Würzburg,https://www.daad.de/en/studying-in-germany/uni...


## 3. Data Processing

### 3.1 Inspecting the Results

In [8]:
programmes.info(), programmes.describe(), programmes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22711 entries, 0 to 22710
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   University   22711 non-null  object
 1   Programme    22711 non-null  object
 2   Degree Type  22711 non-null  object
 3   Location     22711 non-null  object
 4   Detail URL   22711 non-null  object
dtypes: object(5)
memory usage: 887.3+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22711 entries, 0 to 22710
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   University   22711 non-null  object
 1   Programme    22711 non-null  object
 2   Degree Type  22711 non-null  object
 3   Location     22711 non-null  object
 4   Detail URL   22711 non-null  object
dtypes: object(5)
memory usage: 887.3+ KB


(None,
                    University    Programme Degree Type Location  \
 count                   22711        22711       22711    22711   
 unique                    386        10272          13      547   
 top     University of Hamburg  Mathematics      Master   Berlin   
 freq                      368          346       10575     1069   
 
                                                Detail URL  
 count                                               22711  
 unique                                              22711  
 top     https://www.daad.de/en/studying-in-germany/uni...  
 freq                                                    1  ,
 None)

In [9]:
# Split by degree type
bachelor_df = programmes[programmes["Degree Type"] == "Bachelor"]
master_df   = programmes[programmes["Degree Type"] == "Master"]

### 3.2 Save the degree programmes

In [10]:
# Save one Excel file per unique location, for both Bachelor and Master
for location, group in bachelor_df.groupby("Location"):
    safe_loc = location.replace("/", "_").replace("\\", "_")
    group.to_excel(f"{output_dir}/bachelor/{safe_loc}.xlsx", index=False)

for location, group in master_df.groupby("Location"):
    safe_loc = location.replace("/", "_").replace("\\", "_")
    group.to_excel(f"{output_dir}/master/{safe_loc}.xlsx", index=False)

print(f"Saved {len(bachelor_df)} bachelor and {len(master_df)} master programmes")
print(f"Bachelor locations: {bachelor_df['Location']}")
print(f"Master locations: {master_df['Location']}")

Saved 10405 bachelor and 10575 master programmes
Bachelor locations: 1                         Mittweida
2                           Leipzig
3        Idstein, Frankfurt am Main
4                          Würzburg
5                         Pforzheim
                    ...            
22698                        Berlin
22701                     Göttingen
22705                    Düsseldorf
22709                       München
22710                       München
Name: Location, Length: 10405, dtype: object
Master locations: 0              Köln
6        Ingolstadt
7         Stuttgart
10         Nürnberg
11           Weimar
            ...    
22702     Bielefeld
22704      Erlangen
22706    Düsseldorf
22707       Münster
22708       München
Name: Location, Length: 10575, dtype: object
